# LaughTuned — Demo Notebook

Fine-tuning Mistral-7B-Instruct-v0.2 for comedy writing using **DPO** and **KTO**, both implemented from scratch in PyTorch (CS 5788, Cornell).

This notebook demonstrates the pipeline. The main engine lives in the `.py` modules of the repo.

## Step 0 — Environment Setup

Bootstraps the Colab runtime: clones the code repo, installs dependencies, mounts Drive, sets seeds, and verifies the GPU.

In [1]:
# === Colab bootstrap: clone repo, install deps, cd into the code dir ===
# Edit REPO_URL below to point at your GitHub fork before running on Colab.
import os
import subprocess
import sys

REPO_URL = "https://github.com/pcatattacks/laughtuned.git" 
REPO_DIR = "/content/laughtuned"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"Updating existing checkout at {REPO_DIR} ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    os.chdir(REPO_DIR)
    print(f"cwd: {os.getcwd()}")
    print("Installing dependencies (quiet) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
else:
    print("Not on Colab — assuming dependencies are already installed and cwd is the repo root.")

Cloning https://github.com/pcatattacks/laughtuned.git into /content/laughtuned ...
cwd: /content/laughtuned
Installing dependencies (quiet) ...


In [2]:
# === Mount Drive and create the artifact tree ===
from config import CONFIG
from utils.drive_utils import mount_drive, ensure_drive_dirs

mount_drive()
ensure_drive_dirs(CONFIG)

Mounted at /content/drive
[drive_utils] Mounted Drive at /content/drive
[drive_utils] Ready: /content/drive/MyDrive/Colab Notebooks/CS-5788-generative-models/final-project/ (10 subdirectories ensured)


In [3]:
# === Set all random seeds for reproducibility ===
import random
import numpy as np
import torch

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seeded with {SEED}")

Seeded with 42


In [4]:
# === GPU sanity check ===
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | total VRAM: {total_gb:.1f} GB")
else:
    print("No GPU detected. Switch the Colab runtime to T4 or A100 before continuing.")

GPU: Tesla T4 | total VRAM: 15.6 GB


## Step 1 — Load the base model with QLoRA

`load_model_and_tokenizer` does the QLoRA setup: 4-bit NF4 with double quantization, then LoRA adapters (rank 16) on the attention projections. Expect a ~4 GB base load; trainable params should be well under 1% of total.

In [5]:
from models.load_model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(CONFIG)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[load_model] Trainable params: 13,631,488 / 3,765,702,656 (0.362%)
[load_model] GPU: Tesla T4 | allocated: 4.71 GB | reserved: 13.44 GB


## `compute_log_probs` — the shared primitive for DPO and KTO

Both losses score a response by the total log-probability the policy assigns to it given the prompt: `log π(y | x) = Σ_t log π(y_t | x, y_<t)`. The implementation in [`models/log_probs.py`](models/log_probs.py) handles three subtleties:

1. **Shift by one.** A causal LM's output at position `t` predicts the token at `t+1`, so we align `logits[:, :-1]` with `input_ids[:, 1:]` and `label_mask[:, 1:]`.
2. **Prompt tokens contribute zero.** The label mask is 1 only on response tokens (and 0 on prompt tokens *and* padding); after shifting we multiply by it before summing.
3. **Sum, not mean.** DPO and KTO are derived from the total sequence log-probability; length-normalization changes the optimization landscape.

The smoke test below verifies shape, sign, and masking on a synthetic batch.

In [6]:
# === Smoke test for compute_log_probs ===
from models.log_probs import compute_log_probs

# Synthetic batch: 2 examples, 8 tokens each.
# First 4 tokens are "prompt" (mask=0), last 4 are "response" (mask=1).
B, T = 2, 8
device = next(model.parameters()).device
vocab_size = model.config.vocab_size

input_ids_B_T = torch.randint(0, vocab_size, (B, T), device=device)
attention_mask_B_T = torch.ones(B, T, dtype=torch.long, device=device)
label_mask_B_T = torch.zeros(B, T, dtype=torch.long, device=device)
label_mask_B_T[:, T // 2 :] = 1  # response = second half

with torch.no_grad():
    log_probs_B = compute_log_probs(
        model, input_ids_B_T, attention_mask_B_T, label_mask_B_T
    )

# Test 1: shape
assert log_probs_B.shape == (B,), f"expected ({B},), got {tuple(log_probs_B.shape)}"

# Test 2: all values <= 0
assert (log_probs_B <= 0).all(), f"log-probs should be non-positive, got {log_probs_B}"

# Test 3: zeroed mask -> zero output
zero_mask_B_T = torch.zeros_like(label_mask_B_T)
with torch.no_grad():
    zero_log_probs_B = compute_log_probs(
        model, input_ids_B_T, attention_mask_B_T, zero_mask_B_T
    )
assert torch.allclose(
    zero_log_probs_B, torch.zeros_like(zero_log_probs_B)
), f"expected all-zero output for empty mask, got {zero_log_probs_B}"

print("compute_log_probs passed all 3 smoke tests.")
print(f"sample log-probs: {log_probs_B.tolist()}")

compute_log_probs passed all 3 smoke tests.
sample log-probs: [-63.09503936767578, -58.80175018310547]


## Step 2 — Article ingestion with historical backstories

We sample articles from eight Guardian sections (politics, business, technology, sport, culture, science, environment, world). For each, Claude synthesizes a 150-word backstory drawn from the most relevant older Guardian articles on the same topic. Every article ends up with **two** context variants:

- `short_context` — headline + first 400 tokens of today's body
- `long_context` — synthesized backstory + the short context

Holding the comedy-prompt template constant and varying only the context lets us isolate whether richer historical grounding produces better jokes.

Set `GUARDIAN_API_KEY` and `ANTHROPIC_API_KEY` in Colab secrets (left sidebar → 🔑 Secrets) before running the next cell.

In [ ]:
# === Load API keys from Colab secrets (or prompt as fallback) ===
def _load_secret(name: str, required: bool = True) -> str:
    """Try Colab secrets first; fall back to getpass only if required."""
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        return userdata.get(name)
    except Exception:
        if not required:
            return ""
        import getpass
        return getpass.getpass(f"Paste {name}: ")

CONFIG["guardian_api_key"]   = _load_secret("GUARDIAN_API_KEY")
CONFIG["guardian_api_key_2"] = _load_secret("GUARDIAN_API_KEY_2", required=False)
CONFIG["anthropic_api_key"]  = _load_secret("ANTHROPIC_API_KEY")

n_guardian = sum(1 for k in (CONFIG["guardian_api_key"], CONFIG["guardian_api_key_2"]) if k)
print(f"API keys loaded. Guardian keys: {n_guardian} | Anthropic: {'yes' if CONFIG['anthropic_api_key'] else 'no'}")

In [ ]:
# === Smoke test: one round-trip through Guardian + Claude before the full run ===
# Exercises search_section -> search_related -> synthesize_backstory on a single
# article and prints the assembled record. Does not write to disk; safe to run
# before cell 12. Costs ~3 Guardian calls + 1 Claude call (~$0.01).
from datetime import datetime, timedelta, timezone
import anthropic
from data.fetch_articles import (
    search_section,
    search_related,
    synthesize_backstory,
    extract_short_context,
    extract_topic_query,
    BACKSTORY_LOOKBACK_DAYS,
    BACKSTORY_BUFFER_DAYS,
)

guardian_keys = [
    k for k in (CONFIG["guardian_api_key"], CONFIG["guardian_api_key_2"]) if k
]
claude = anthropic.Anthropic(api_key=CONFIG["anthropic_api_key"])
office = CONFIG.get("guardian_production_office")
assert guardian_keys, "no Guardian keys loaded"
assert CONFIG["anthropic_api_key"], "no Anthropic key loaded"

# 1. Pull a few recent politics articles, take the first one
today = datetime.now(timezone.utc).date()
section_articles = search_section(
    guardian_keys,
    section="politics",
    from_date=(today - timedelta(days=30)).isoformat(),
    to_date=today.isoformat(),
    page_size=3,
    production_office=office,
)
assert section_articles, "section search returned no articles"
raw = section_articles[0]
headline = (raw.get("fields") or {}).get("headline") or raw.get("webTitle", "")
print(f"Article: {headline!r}")
print(f"  published: {raw.get('webPublicationDate')}")

# 2. Build short context + topic query
short_context = extract_short_context(raw, tokenizer)
topic_query = extract_topic_query(raw)
print(f"  short_context: {len(short_context)} chars")
print(f"  topic_query:   {topic_query[:80]}")

# 3. Related search (top 3 by relevance within the lookback window)
published_date = datetime.fromisoformat(
    raw["webPublicationDate"].replace("Z", "+00:00")
).date()
older = search_related(
    guardian_keys,
    topic_query=topic_query,
    from_date=(published_date - timedelta(days=BACKSTORY_LOOKBACK_DAYS)).isoformat(),
    to_date=(published_date - timedelta(days=BACKSTORY_BUFFER_DAYS)).isoformat(),
    production_office=office,
)
older = [o for o in older if o["id"] != raw["id"]]
print(f"\nHistorical articles ({len(older)}):")
for o in older:
    o_head = (o.get("fields") or {}).get("headline") or o.get("webTitle", "")
    print(f"  - {o.get('webPublicationDate', '?')[:10]}  {o_head}")

# 4. Backstory synthesis via Claude
backstory = synthesize_backstory(claude, CONFIG["judge_model"], raw, older, tokenizer)
print(f"\nBackstory ({len(backstory)} chars):")
print(backstory[:400] + ("..." if len(backstory) > 400 else ""))

# 5. Final long_context shape
long_context = f"{backstory}\n\n{short_context}" if backstory else short_context
print(
    f"\nlong_context: {len(long_context)} chars "
    f"(+{len(long_context) - len(short_context)} vs short_context)"
)
print("\nSmoke test passed.")

In [ ]:
# === Ingest articles (idempotent: resumes from disk if interrupted) ===
from data.fetch_articles import ingest_articles

articles = ingest_articles(CONFIG, tokenizer)

print(f"\nTotal articles: {len(articles)}")
print(f"Sections covered: {sorted({a['section'] for a in articles})}")

sample = articles[0]
print(f"\n--- Sample article ---")
print(f"headline:      {sample['headline']}")
print(f"section:       {sample['section']}")
print(f"published:     {sample['published_at']}")
print(f"older refs:    {len(sample['older_article_ids'])} historical articles used")
print(f"short_context: {len(sample['short_context'])} chars")
print(f"long_context:  {len(sample['long_context'])} chars  "
      f"(+{len(sample['long_context']) - len(sample['short_context'])} from backstory)")
print(f"\nshort_context preview:\n{sample['short_context'][:300]}...")
print(f"\nlong_context preview:\n{sample['long_context'][:400]}...")